# Lab - Authorize Actions in Application Code

Three tasks on where a rule belongs. Some rules shape behaviour and live in the prompt; a rule that must always hold is an authorization check in your code, against your own data.

| Task | What you learn |
| --- | --- |
| 1 | Role and scope in a versioned prompt config - and a test that it declines |
| 2 | authorize() blocks an unverified refund before any request leaves |
| 3 | verify() checks the email against the order; only then the refund passes |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The store's own record of the order - what the application checks against.
ORDERS = {
    "ORD-12345678": {"item": "jacket", "email": "alex@example.com"},
}
REFUND_TEXT = "I want a refund for the jacket I bought 20 days ago."

---

## Task 1 - Role and scope belong in the prompt

In [ ]:
# ============================================================
# TASK 1 - Role and scope belong in the prompt
# ============================================================
#
# WHAT TO DO
#   Keep the system prompt in one configuration entry, with a
#   version label, and finish its scope: what ShopAssist helps
#   with, and what it does with everything else. Then test the
#   scope with a request it should decline.
#
# WHY IT MATTERS
#   Role, tone, scope and the style of an answer follow the
#   system prompt reliably - that is what it does well. Kept as
#   one versioned piece of configuration, rather than strings
#   scattered through the code, a prompt can be reviewed,
#   changed and traced: you know which version gave which
#   answer.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Let's make it a real system prompt".
#
# HOW TO DO IT
#   Three blanks:
#
#       "Help only with orders, returns and refunds."
#       "For anything else, say politely that you cannot help with it."
#       SYSTEM   the assembled prompt, sent with every request
# ============================================================

# TODO: replace each ... below with the value named beside it
PROMPT_CONFIG = {
    "version": "rules-2",
    "role": "You are ShopAssist, an online store's support assistant.",
    "scope": ...,     # "Help only with orders, returns and refunds."
    "decline": ...,   # "For anything else, say politely that you cannot help with it."
    "policy": "Returns are accepted within 30 days of delivery.",
    "style": "Answer in one or two sentences.",
}
SYSTEM = " ".join(str(PROMPT_CONFIG[key]) for key in
                  ("role", "scope", "decline", "policy", "style"))

REQUESTS = 0   # every request that leaves for Claude


def ask(messages):
    global REQUESTS
    REQUESTS += 1
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=...,   # SYSTEM
        messages=messages,
    )
    return next(b.text for b in response.content if b.type == "text")

poem_reply = ask([{"role": "user", "content": "Write me a short poem about jackets."}])
print("prompt", PROMPT_CONFIG["version"], "->", poem_reply)

check("prompt_scope", config=PROMPT_CONFIG, system=SYSTEM, poem_reply=poem_reply)

---

## Task 2 - Authorize the action in code

In [ ]:
# ============================================================
# TASK 2 - Authorize the action in code
# ============================================================
#
# WHAT TO DO
#   Finish authorize(): a refund request from a session that is
#   not verified gets a fixed answer from your code. say() asks
#   authorize() first, writes every decision to the audit log,
#   and only an allowed message goes on to Claude.
#
# WHY IT MATTERS
#   "Never discuss a refund before the customer confirms the
#   email" could be one more line in the prompt. But a prompt is
#   a request, not a guarantee: a customer who insists, or claims
#   the check was done, may talk the model into it. An
#   authorization check in code runs before the request leaves,
#   and no conversation can change it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Here is the shape of the gate".
#
# HOW TO DO IT
#   Three blanks:
#
#       "refund"     the action that needs a verified customer
#       "verified"   the session flag authorize() reads
#       refusal      what say() returns instead of asking Claude
# ============================================================

AUDIT = []   # provided: one entry per decision

# TODO: replace each ... below with the value named beside it
def authorize(session, text):
    """None when the message may go to Claude, otherwise the answer to give instead."""
    if ... in text.lower() and not session[...]:   # "refund", "verified"
        return "Please confirm the email of your order first."
    return None


def say(session, text):
    refusal = authorize(session, text)
    AUDIT.append({"text": text, "decision": "blocked" if refusal else "allowed"})
    if refusal is not None:
        print("gate:  ", refusal)
        return ...   # refusal - Claude never sees this message
    session["history"].append({"role": "user", "content": text})
    reply = ask(session["history"])
    session["history"].append({"role": "assistant", "content": reply})
    print("Claude:", reply)
    return reply

session = {"verified": False, "history": []}   # provided: a new customer's session
AUDIT.clear()
try:
    sent_before = REQUESTS
    say(session, REFUND_TEXT)
    sent_after = REQUESTS
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("requests sent for that turn:", sent_after - sent_before)

check("authorize", authorize=authorize, sent_before=sent_before, sent_after=sent_after,
      audit=AUDIT)

---

## Task 3 - Verify against the order, then pass

In [ ]:
# ============================================================
# TASK 3 - Verify against the order, then pass
# ============================================================
#
# WHAT TO DO
#   Finish verify(): the session becomes verified only when the
#   email matches the one stored with the order. Then run the
#   whole path: a refund before any check, a mistyped email, the
#   right email, and the refund again.
#
# WHY IT MATTERS
#   In a real system the application checks the email against
#   the order - against its own data, not against what the
#   customer claims. A wrong email changes nothing; the right
#   one flips the session's flag, and only then does the same
#   message pass the gate and reach Claude. The audit log shows
#   every step.
#
# WHERE TO SEE IT IN THE LECTURE
#   "In a real system, the application would check it against
#   the order".
#
# HOW TO DO IT
#   Three blanks:
#
#       order_id   look the order up by its ID
#       "email"    the email stored with the order
#       True       the session is verified now
# ============================================================

# TODO: replace each ... below with the value named beside it
def verify(session, order_id, email):
    order = ORDERS.get(...)   # order_id
    if order is not None and order[...] == email.strip().lower():   # "email"
        session["verified"] = ...   # True
    return session["verified"]

session = {"verified": False, "history": []}   # provided: the whole path, from the start
AUDIT.clear()
try:
    say(session, REFUND_TEXT)
    print("verify with a typo:", verify(session, "ORD-12345678", "alex@exemple.com"))
    say(session, REFUND_TEXT)
    print("verify, right email:", verify(session, "ORD-12345678", "alex@example.com"))
    refund_reply = say(session, REFUND_TEXT)
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for entry in AUDIT:
    print(entry["decision"], "-", entry["text"])

check("verify_email", session=session, audit=AUDIT, refund_reply=refund_reply)

---

## Done

Put role, tone and scope in the system prompt, versioned as configuration. Put every rule that must always hold in code - an authorization check against your own data, before the request leaves your application.